# IndabaX Colab worker

Trains the queued jobs one after the other on Colab's GPU, to save Kaggle GPU quota. The queue is the private Kaggle dataset `cocofioren/indabax-colab-jobs`; each job's results go to a private dataset `cocofioren/indabax-colab-<job>`. Jobs whose results already exist are skipped, so running it again after a disconnect resumes the queue.

1. Runtime > Change runtime type > **T4 GPU**.
2. The `KAGGLE_API_TOKEN` secret (key icon, left sidebar) must allow this notebook to read it.
3. Runtime > **Run all**, and keep the tab open.


In [ ]:
!nvidia-smi -L
!pip -q install -U kaggle segmentation-models-pytorch

In [ ]:
import os, pathlib
from google.colab import userdata
tok = userdata.get('KAGGLE_API_TOKEN').strip()
d = pathlib.Path.home() / '.kaggle'; d.mkdir(exist_ok=True)
(d / 'access_token').write_text(tok); os.chmod(d / 'access_token', 0o600)
os.environ['KAGGLE_API_TOKEN'] = tok
print('token loaded from Colab secrets')

In [ ]:
!mkdir -p /content/data /content/out
!kaggle competitions download -c hackathon-IndabaX-MDG-2026 -f train.npz -p /content/data
!kaggle competitions download -c hackathon-IndabaX-MDG-2026 -f test.npz -p /content/data
!cd /content/data && for z in *.zip; do [ -e "$z" ] && unzip -o -q "$z" && rm "$z"; done; ls -la /content/data

In [ ]:
import glob, json, os, shutil, subprocess
def upload(out, name):
    up = f"/content/upload/{name}"; shutil.rmtree(up, ignore_errors=True); os.makedirs(up)
    for f in glob.glob(f"{out}/*_testprob.npy") + glob.glob(f"{out}/*.json"):
        shutil.copy(f, up)
    jsons = glob.glob(f"{out}/*.json")
    for jf in jsons:
        blends = json.load(open(jf)).get("epoch_blends", {})
        if blends:  # keep only the best last-k validation file (each is about 280 MB)
            k = max(blends, key=lambda b: blends[b]["val_fss"]); tag = os.path.basename(jf)[:-5]
            shutil.copy(f"{out}/{tag}_{k}_valprob.npy", up); print("best epoch blend", k, blends[k]["val_fss"])
    if not jsons:  # the run stopped before its summary: keep every validation file and the weights
        for f in glob.glob(f"{out}/*_valprob.npy") + glob.glob(f"{out}/*.pt"):
            shutil.copy(f, up)
    json.dump({"title": name, "id": f"cocofioren/{name}", "licenses": [{"name": "CC0-1.0"}]}, open(f"{up}/dataset-metadata.json", "w"))
    print(sorted(os.listdir(up)), flush=True)
    if subprocess.run(f"kaggle datasets create -p {up} -q", shell=True).returncode != 0:  # private by default
        subprocess.run(f"kaggle datasets version -p {up} -m rerun -q", shell=True)
    shutil.rmtree(up, ignore_errors=True)

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
JOBS = "cocofioren/indabax-colab-jobs"
def results_name(slug): return ("indabax-colab-" + slug.removeprefix("indabax-"))[:50]
def uploaded(slug):
    r = subprocess.run(f"kaggle datasets status cocofioren/{results_name(slug)}", shell=True, capture_output=True, text=True)
    return "ready" in r.stdout.lower()
done, failed = set(), set()
while True:
    shutil.rmtree("/content/jobs", ignore_errors=True)
    if subprocess.run(f"kaggle datasets download -d {JOBS} -p /content/jobs --unzip -o -q", shell=True).returncode != 0:
        print("could not download the job queue"); break
    queue = json.load(open("/content/jobs/jobs.json"))
    todo = [s for s in queue if s not in done and s not in failed and not uploaded(s)]
    print("jobs left:", todo, flush=True)
    if not todo:
        print("all jobs done: the runtime can be closed (Runtime > Disconnect and delete runtime)"); break
    slug = todo[0]; out = f"/content/out/{slug}"; shutil.rmtree(out, ignore_errors=True)
    script = open(f"/content/jobs/{slug}.py").read().replace("--out /content/out", f"--out {out}", 1)
    open("/content/job.py", "w").write(script)
    print("=" * 20, "running", slug, flush=True)
    rc = subprocess.run("set -o pipefail; python -u /content/job.py 2>&1 | grep --line-buffered -v Warning",
                        shell=True, executable="bash").returncode
    if not glob.glob(f"{out}/*_testprob.npy"):
        print("job failed:", slug, "exit code", rc, flush=True); failed.add(slug); continue
    if rc != 0: print("job stopped early:", slug, "exit code", rc, "- uploading what it saved", flush=True)
    upload(out, results_name(slug)); done.add(slug); shutil.rmtree(out, ignore_errors=True)
